# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [4]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are a healthy, convenient fruit.
    
    Key points:
    - Nutrients: good source of potassium, vitamin C, vitamin B6, and fiber.
    - Benefits: supports heart health, digestion, and provides quick energy.
    - Glycemic load: moderate; ripe bananas have more sugar but still fit in a balanced diet.
    - Considerations: portion size if watching carbs/sugar; people with kidney disease may need to monitor potassium intake.
    - Typical serving: 1 medium banana (~118 g) ~105 calories.
    
    Pair with protein or fat (e.g., peanut butter) for staying power.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [5]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are healthy in moderation. Key points:

- Good nutrients: potassium (~400 mg), vitamin B6, vitamin C, fiber (about 3 g), and about 14 g sugar with ~105 calories in a medium banana.
- Benefits: supports heart health, energy for workouts, digestion (prebiotic fiber), and satiety.
- Glycemic index: moderate; ripe bananas have more sugar but still fit into a balanced diet.
- Considerations: portion control if watching calories or sugar; people with kidney disease may need potassium limits—consult a doctor.
- Tips: eat whole for fiber, pair with protein/fat (e.g., peanut butter) to balance blood sugar; refrigerate ripe bananas if you want to slow browning.

Want a quick plan for daily intake based on your goals?

_Good Job!_